# TP 2 bis : Comprendre Prefect et son interface (CORRECTION)

**BUT INFO3 Développeur IA, R5.C.06, DATA5, Semaine 45**
M. Sakkari, IUT de La Rochelle

> **Version enseignant :** codes complétés et réponses aux questions.

## Dans ce TP, on apprend à...
- expliquer ce qui se passe quand on lance un flux ;
- lire une exécution dans l'interface : états, tâches, journal, paramètres ;
- retrouver la cause d'un échec ;
- comprendre les états d'une exécution et les relances ;
- lancer un flux depuis l'interface, avec un paramètre ;
- lire et piloter une planification dans l'interface ;
- garder l'historique des changements d'un fichier ;
- publier un résumé dans l'interface.

> **Comment travailler :**
> - Une cellule « Démo » est déjà écrite : lancez-la et lisez-la.
> - Une cellule « À vous » est à compléter : remplacez les `...` par votre code.
> - Les questions « Dans l'interface » se font dans le navigateur. Notez la réponse sous la question.

## Le point de départ
Vous partez de la correction du TP 2. Le dossier du TP contient déjà :

| Fichier | Contenu |
|---|---|
| `TP2_Prefect_correction.ipynb` | la correction du TP 2 |
| `serveur_prefect.py` | le script de la partie F du TP 2 (flux lancé toutes les 30 secondes) |
| `surveillance_prefect.py` | le script de la partie G du TP 2 (surveillance d'un fichier) |
| `etl.py`, `outils.py`, `data/` | le pipeline et ses outils |

## Avant de commencer
Dans ce TP, **le serveur Prefect reste lancé du début à la fin**. Ainsi, tout ce que vous exécutez apparaît dans l'interface.

1. Ouvrez un terminal dans le dossier du TP, avec l'environnement `.venv` activé.
2. Tapez : `prefect server start`. Laissez ce terminal ouvert.
3. Ouvrez l'interface dans le navigateur : http://127.0.0.1:4200
4. Ouvrez un second terminal dans le dossier du TP. Tapez une seule fois (sans point à la fin) : `prefect config set PREFECT_API_URL=http://127.0.0.1:4200/api`
5. Dans ce notebook, redémarrez le noyau, puis exécutez les deux cellules ci-dessous.

In [22]:
# Cette cellule prépare le notebook. Il n'y a rien à compléter.
import json
import os
import sqlite3
import time
import urllib.request
from datetime import datetime

import pandas as pd
from prefect import flow, get_run_logger, task
from prefect.runtime import flow_run
from prefect.settings import PREFECT_API_URL

import etl
import outils

# Windows affiche parfois une erreur sans gravité quand une connexion se ferme : on la cache
import logging
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

print("Les outils sont chargés.")

Les outils sont chargés.


In [23]:
# Cette cellule vérifie que le serveur Prefect répond. Il n'y a rien à compléter.
adresse = PREFECT_API_URL.value()
print("Adresse du serveur Prefect :", adresse)
try:
    reponse = urllib.request.urlopen(adresse + "/health", timeout=5).read().decode()
    print("Le serveur Prefect répond :", reponse)
except Exception as erreur:
    print("Le serveur ne répond pas. Lancez `prefect server start` dans un terminal, puis relancez cette cellule.")
    print("Détail :", erreur)

Adresse du serveur Prefect : http://127.0.0.1:4200/api
Le serveur Prefect répond : true


**Lecture :** la deuxième cellule affiche `http://127.0.0.1:4200/api`, puis `true`.
Si l'adresse est autre chose, retapez la commande `prefect config set ...` de l'étape 4, redémarrez le noyau et recommencez.

## Partie A : Ce qui se passe quand on lance un flux
Quand vous lancez un flux, quatre éléments interviennent :

```
 1. Votre code Python         2. Le serveur Prefect          3. La base de Prefect
    (le notebook ou un   ──►     (le programme lancé    ──►     (le fichier qui garde
     script)                      par prefect server            tout l'historique)
                                  start)
                                       ▲
                                       │ demande ce qui est enregistré
                                       │
                                4. L'interface web
                                   (le navigateur, à l'adresse http://127.0.0.1:4200)
```

1. **Votre code Python** exécute le flux et ses tâches. À chaque changement d'état (démarrage, réussite, échec), il envoie un message au serveur.
2. **Le serveur Prefect** reçoit ces messages et les enregistre dans sa base.
3. **La base de Prefect** garde les exécutions, leurs états, leurs journaux et leurs paramètres. Ce n'est pas la base `cafe.db` du café.
4. **L'interface web** demande au serveur ce qu'il a enregistré et l'affiche. Elle ne fait pas tourner votre code.

Si le serveur est arrêté, votre code ne peut plus envoyer ses messages. C'est pourquoi il reste lancé pendant tout le TP.

### Démo : A.1 lancer un flux et noter le nom de l'exécution
Chaque exécution reçoit un nom, choisi au hasard par Prefect.

In [24]:
# Cette cellule lance un flux et affiche le nom de son exécution. Il n'y a rien à compléter.

@task
def dire_bonjour(prenom):
    return "Bonjour " + prenom


@flow
def accueil():
    dire_bonjour("Niort")
    return flow_run.name


nom_execution = accueil()
print("Le nom de cette exécution est :", nom_execution)

13:46:48.639 | INFO    | Flow run 'white-macaque' - Beginning flow run 'white-macaque' for flow 'accueil'

13:46:48.641 | INFO    | Flow run 'white-macaque' - View at http://127.0.0.1:4200/runs/flow-run/f1dfaef3-55ab-497a-899a-13c3c6c9bb5e

13:46:48.649 | INFO    | Task run 'dire_bonjour-fd8' - Finished in state Completed()

13:46:49.644 | INFO    | Flow run 'white-macaque' - Finished in state Completed()

Le nom de cette exécution est : white-macaque


### À vous : A.2 retrouver l'exécution dans l'interface
**Dans l'interface :**
1. Cliquez sur « Runs » dans le menu de gauche.
2. Tapez le nom de l'exécution dans la case « Search by flow run name ».
3. Cliquez sur l'exécution.

**Questions :**
1. Quel est l'état de l'exécution ?
2. Quel est le nom du flux ?
3. Combien de tâches (« Task runs ») contient-elle ?

*Votre réponse :* 
1. `Completed`.
2. `accueil`.
3. Une tâche : `dire_bonjour`.

## Partie B : Lire une exécution
### Démo : B.1 un flux avec trois étapes
Ce flux a un **paramètre** : le nom du fichier à traiter, comme dans le cours (« Vocabulaire (2/6) », flux de travail). Il écrit un message dans le journal, et une de ses étapes est plus lente que les autres.

In [25]:
# Cette cellule lance un flux en trois étapes. Il n'y a rien à compléter.

@task
def lire_fichier():
    time.sleep(1)
    return "ok"


@task
def nettoyer_lignes():
    time.sleep(4)
    return "ok"


@task
def ecrire_resultat():
    time.sleep(1)
    return "ok"


@flow
def flux_trois_etapes(fichier: str = "commandes.csv"):
    journal = get_run_logger()
    journal.info("Le flux démarre avec le fichier %s.", fichier)
    lire_fichier()
    nettoyer_lignes()
    ecrire_resultat()
    return flow_run.name


print("Le nom de cette exécution est :", flux_trois_etapes())

13:46:57.739 | INFO    | Flow run 'scarlet-harrier' - Beginning flow run 'scarlet-harrier' for flow 'flux-trois-etapes'

13:46:57.740 | INFO    | Flow run 'scarlet-harrier' - View at http://127.0.0.1:4200/runs/flow-run/aa8a59cd-b98f-4da5-92dc-b228035a2594

13:46:57.742 | INFO    | Flow run 'scarlet-harrier' - Le flux démarre avec le fichier commandes.csv.

13:46:58.753 | INFO    | Task run 'lire_fichier-c9a' - Finished in state Completed()

13:47:02.762 | INFO    | Task run 'nettoyer_lignes-5fb' - Finished in state Completed()

13:47:03.768 | INFO    | Task run 'ecrire_resultat-a86' - Finished in state Completed()

13:47:04.745 | INFO    | Flow run 'scarlet-harrier' - Finished in state Completed()

Le nom de cette exécution est : scarlet-harrier


### À vous : B.2 lire une exécution
**Dans l'interface :**
1. Cliquez sur « Flows » dans le menu de gauche. Une liste de flux apparaît. L'interface remplace les tirets bas par des tirets : cherchez `flux-trois-etapes`.
2. Dans la colonne « Last Run » de cette ligne, cliquez sur le nom de l'exécution. C'est le nom que la cellule B.1 vient d'afficher (« Le nom de cette exécution est : ... »). Vous êtes alors sur la page de l'exécution.

La page de l'exécution montre, en haut, un graphe des tâches dans le temps : chaque barre est une tâche. Dessous, plusieurs onglets :
- L'onglet « Details » donne les informations générales de l'exécution.
- L'onglet « Logs » montre le journal.
- L'onglet « Task Runs » liste les tâches.
- L'onglet « Parameters » montre les paramètres reçus.

**Questions :**
1. Quelle tâche dure le plus longtemps ? Combien de temps ?
2. Quelle valeur a le paramètre `fichier` ?
3. Quelle ligne du journal vient de `journal.info` ?

*Votre réponse :* 
1. `nettoyer_lignes`, environ 4 secondes.
2. `commandes.csv`.
3. « Le flux démarre avec le fichier commandes.csv. »

### Démo : B.3 un flux qui échoue
Cette cellule crée un fichier de commandes abîmé, puis lance un flux sur ce fichier. Le flux échoue : c'est voulu.

In [26]:
# Cette cellule lance un flux qui échoue. Il n'y a rien à compléter.
with open("data/commandes_casse.csv", "w", encoding="utf8") as fichier:
    fichier.write("id;client;montant\n1;2;3\n")


@task
def controler_source(fichier):
    problemes = outils.verifier_source("data/" + fichier, outils.COLONNES_COMMANDES)
    if problemes:
        raise ValueError("Fichier refusé : " + ", ".join(problemes))


@task
def lancer_pipeline(fichier):
    con = sqlite3.connect(etl.DB)
    etl.pipeline(con, fichier, flow_run.name, declencheur="prefect")
    con.close()


@flow
def flux_controle(fichier):
    controler_source(fichier)
    lancer_pipeline(fichier)


etat = flux_controle("commandes_casse.csv", return_state=True)
print("Le flux a fini à l'état :", etat.name)

13:47:07.933 | INFO    | Flow run 'emerald-jackalope' - Beginning flow run 'emerald-jackalope' for flow 'flux-controle'

13:47:07.935 | INFO    | Flow run 'emerald-jackalope' - View at http://127.0.0.1:4200/runs/flow-run/2f1c74cc-202c-433d-bb3c-69e3534ced8f

13:47:07.943 | ERROR   | Task run 'controler_source-99e' - Task run failed with exception: ValueError('Fichier refusé : colonnes manquantes : id_commande, id_client, sku, quantite, date_commande, montant_ttc, statut')
Traceback (most recent call last):
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1058, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1755, in run_task_sync
    engine.call_task_fn(txn)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1075, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_31708\970734793.py", line 10, in controler_source
    raise ValueError("Fichier refusé : " + ", ".join(problemes))
ValueError: Fichier refusé : colonnes manquantes : id_commande, id_client, sku, quantite, date_commande, montant_ttc, statut

13:47:07.948 | ERROR   | Task run 'controler_source-99e' - Finished in state Failed('Task run encountered an exception ValueError: Fichier refusé : colonnes manquantes : id_commande, id_client, sku, quantite, date_commande, montant_ttc, statut')

13:47:08.938 | ERROR   | Flow run 'emerald-jackalope' - Encountered exception during execution: ValueError('Fichier refusé : colonnes manquantes : id_commande, id_client, sku, quantite, date_commande, montant_ttc, statut')
Traceback (most recent call last):
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 1339, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 2133, in run_flow_sync
    engine.call_flow_fn()
    ~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 1361, in call_flow_fn
    result = call_with_parameters(self.flow.fn, self.parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_31708\970734793.py", line 22, in flux_controle
    controler_source(fichier)
    ~~~~~~~~~~~~~~~~^^^^^^^^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\tasks.py", line 1240, in __call__
    return run_task(
        task=self,
    ...<2 lines>...
        return_type=return_type,
    )
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1988, in run_task
    return run_task_sync(**kwargs)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1758, in run_task_sync
    return engine.state if return_type == "state" else engine.result()
                                                       ~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 622, in result
    raise self._raised
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1058, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1755, in run_task_sync
    engine.call_task_fn(txn)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1075, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_31708\970734793.py", line 10, in controler_source
    raise ValueError("Fichier refusé : " + ", ".join(problemes))
ValueError: Fichier refusé : colonnes manquantes : id_commande, id_client, sku, quantite, date_commande, montant_ttc, statut

13:47:09.029 | INFO    | Flow run 'emerald-jackalope' - Finished in state Failed('Flow run encountered an exception: ValueError: Fichier refusé : colonnes manquantes : id_commande, id_client, sku, quantite, date_commande, montant_ttc, statut')

Le flux a fini à l'état : Failed


**Dans l'interface :** ouvrez l'exécution du flux `flux_controle` à l'état « Failed ».

**Questions :**
1. Quelle tâche a échoué ?
2. Quel est le message d'erreur ? Où le lisez-vous ?
3. La tâche `lancer_pipeline` a-t-elle tourné ? Comment le voyez-vous ?

*Votre réponse :* 
1. `controler_source`.
2. « Fichier refusé : colonnes manquantes : ... ». On le lit dans l'onglet « Logs », ou dans l'état « Failed » en haut de la page.
3. Non. Elle n'apparaît pas dans la liste des tâches : le flux s'est arrêté avant.

## Partie C : Les états et les relances
Une exécution passe par plusieurs **états**. Prefect affiche toujours l'état courant.

| État | Ce que ça veut dire |
|---|---|
| `Scheduled` | l'exécution est prévue pour plus tard |
| `Pending` | elle est sur le point de démarrer |
| `Running` | elle s'exécute |
| `Completed` | elle a réussi |
| `Failed` | elle a échoué à cause d'une erreur du code (une exception) |
| `Crashed` | elle s'est arrêtée de façon anormale, sans erreur du code : machine éteinte, processus tué |
| `Cancelled` | quelqu'un l'a annulée |

### Démo : C.1 une relance qui réussit

In [27]:
# Cette cellule lance un flux dont la tâche est relancée. Il n'y a rien à compléter.

@task(retries=3, retry_delay_seconds=2)
def appeler_source_robuste():
    return outils.source_instable()      # échoue aux essais 1 et 2, puis répond


@flow
def flux_source_robuste():
    print("Le nom de cette exécution est :", flow_run.name)
    return appeler_source_robuste()


outils.reinitialiser_source_instable()
etat = flux_source_robuste(return_state=True)
print("Le flux a fini à l'état :", etat.name)

13:47:16.420 | INFO    | Flow run 'majestic-wolf' - Beginning flow run 'majestic-wolf' for flow 'flux-source-robuste'

13:47:16.422 | INFO    | Flow run 'majestic-wolf' - View at http://127.0.0.1:4200/runs/flow-run/b064aaf2-7e51-4ef6-bdb7-56d52bc67792

Le nom de cette exécution est : majestic-wolf


13:47:16.430 | INFO    | Task run 'appeler_source_robuste-ac5' - Task run failed with exception: ConnectionError('La source est indisponible (essai 1).') - Retry 1/3 will start 2 second(s) from now

13:47:18.434 | INFO    | Task run 'appeler_source_robuste-ac5' - Task run failed with exception: ConnectionError('La source est indisponible (essai 2).') - Retry 2/3 will start 2 second(s) from now

13:47:20.446 | INFO    | Task run 'appeler_source_robuste-ac5' - Finished in state Completed()

13:47:21.426 | INFO    | Flow run 'majestic-wolf' - Finished in state Completed()

Le flux a fini à l'état : Completed


**Dans l'interface :** ouvrez l'exécution dont la cellule vient d'afficher le nom (menu « Flows », ligne `flux-source-robuste`, colonne « Last Run »), puis l'onglet « Logs ».

**Questions :**
1. Combien de fois la tâche a-t-elle été essayée ?
2. Quel est l'état final du flux ?
3. Quel temps sépare deux essais ?

*Votre réponse :* 
1. Trois fois : deux échecs, puis une réussite.
2. `Completed`.
3. Deux secondes (`retry_delay_seconds=2`).

### À vous : C.2 une relance qui ne suffit pas
**À faire :** ne relancez la tâche qu'une seule fois, avec 2 secondes d'attente. La source échoue deux fois : le flux doit donc échouer.

In [28]:
# Objectif : relancer une seule fois la tâche qui échoue.
# Résultat attendu : le flux finit à l'état Failed, après deux essais.

# Étape 1 : le décorateur de la tâche, avec 1 relance et 2 secondes d'attente
@task(retries=1, retry_delay_seconds=2)
def appeler_source_limitee():
    return outils.source_instable()


@flow
def flux_une_relance():
    print("Le nom de cette exécution est :", flow_run.name)
    return appeler_source_limitee()


outils.reinitialiser_source_instable()
etat = flux_une_relance(return_state=True)
print("Le flux a fini à l'état :", etat.name)

13:48:05.738 | INFO    | Flow run 'raspberry-gopher' - Beginning flow run 'raspberry-gopher' for flow 'flux-une-relance'

13:48:05.744 | INFO    | Flow run 'raspberry-gopher' - View at http://127.0.0.1:4200/runs/flow-run/d4f08f85-c4d6-4a9f-93af-dcc3eec6fee5

Le nom de cette exécution est : raspberry-gopher


13:48:05.754 | INFO    | Task run 'appeler_source_limitee-a9e' - Task run failed with exception: ConnectionError('La source est indisponible (essai 1).') - Retry 1/1 will start 2 second(s) from now

13:48:07.761 | ERROR   | Task run 'appeler_source_limitee-a9e' - Task run failed with exception: ConnectionError('La source est indisponible (essai 2).') - Retries are exhausted
Traceback (most recent call last):
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1058, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1755, in run_task_sync
    engine.call_task_fn(txn)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1075, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_31708\1318619321.py", line 7, in appeler_source_limitee
    return outils.source_instable()
           ~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\data5\S45_Automatisation\TP2bis_Prefect\outils.py", line 52, in source_instable
    raise ConnectionError(f"La source est indisponible (essai {essai}).")
ConnectionError: La source est indisponible (essai 2).

13:48:07.765 | ERROR   | Task run 'appeler_source_limitee-a9e' - Finished in state Failed('Task run encountered an exception ConnectionError: La source est indisponible (essai 2).')

13:48:08.748 | ERROR   | Flow run 'raspberry-gopher' - Encountered exception during execution: ConnectionError('La source est indisponible (essai 2).')
Traceback (most recent call last):
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 1339, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 2133, in run_flow_sync
    engine.call_flow_fn()
    ~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 1361, in call_flow_fn
    result = call_with_parameters(self.flow.fn, self.parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_31708\1318619321.py", line 13, in flux_une_relance
    return appeler_source_limitee()
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\tasks.py", line 1240, in __call__
    return run_task(
        task=self,
    ...<2 lines>...
        return_type=return_type,
    )
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1988, in run_task
    return run_task_sync(**kwargs)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1758, in run_task_sync
    return engine.state if return_type == "state" else engine.result()
                                                       ~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 622, in result
    raise self._raised
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1058, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1755, in run_task_sync
    engine.call_task_fn(txn)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1075, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_31708\1318619321.py", line 7, in appeler_source_limitee
    return outils.source_instable()
           ~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\data5\S45_Automatisation\TP2bis_Prefect\outils.py", line 52, in source_instable
    raise ConnectionError(f"La source est indisponible (essai {essai}).")
ConnectionError: La source est indisponible (essai 2).

13:48:08.805 | INFO    | Flow run 'raspberry-gopher' - Finished in state Failed('Flow run encountered an exception: ConnectionError: La source est indisponible (essai 2).')

Le flux a fini à l'état : Failed


**Dans l'interface :** ouvrez l'exécution dont la cellule vient d'afficher le nom (menu « Flows », ligne `flux-une-relance`, colonne « Last Run »).

**Question :** combien d'essais voyez-vous dans le journal ? Quel est l'état de la tâche ?

*Votre réponse :* deux essais, tous deux en échec. La tâche et le flux sont à l'état `Failed`.

## Partie D : Déploiements et paramètres
Jusqu'ici, c'est le notebook qui lance les flux. Avec `.serve()`, vous **publiez** un flux : Prefect le garde sous un nom, avec ses réglages, et l'interface peut le lancer.

Un flux publié s'appelle un **déploiement** : un flux avec un nom et des réglages (planification, paramètres, étiquettes).

Quand l'interface « lance » un déploiement, elle demande seulement au serveur de prévoir une exécution. C'est le **script publié**, qui tourne dans votre terminal, qui exécute le code. Si ce script est arrêté, rien ne s'exécute.

### À vous : D.1 publier un flux qui reçoit un paramètre
**À faire :** complétez le script `serveur_parametre.py`.
1. Le flux `pipeline_parametre(fichier)` contrôle le fichier, puis lance le pipeline. Les deux tâches sont données.
2. Le flux est publié sous le nom `pipeline-parametre`.

Le paramètre du flux s'écrit `fichier: str = "commandes_increment.csv"` : un nom, un type et une valeur par défaut. L'interface propose ce paramètre quand on lance le flux.

In [12]:
%%writefile serveur_parametre.py
# Objectif : publier un flux qui reçoit le nom du fichier en paramètre.
# Résultat attendu : dans l'interface, le déploiement « pipeline-parametre » permet de choisir le fichier à chaque lancement.
import sqlite3

from prefect import flow, task
from prefect.runtime import flow_run

import etl
import outils


@task
def controler_source(fichier):
    problemes = outils.verifier_source("data/" + fichier, outils.COLONNES_COMMANDES)
    if problemes:
        raise ValueError("Fichier refusé : " + ", ".join(problemes))


@task
def lancer_pipeline(fichier):
    con = sqlite3.connect(etl.DB)
    etl.pipeline(con, fichier, flow_run.name, declencheur="prefect")
    con.close()


@flow
def pipeline_parametre(fichier: str = "commandes_increment.csv"):
    # Étape 1 : contrôler le fichier, puis lancer le pipeline
    controler_source(fichier)
    lancer_pipeline(fichier)


if __name__ == "__main__":
    # Étape 2 : publier le flux sous le nom « pipeline-parametre »
    pipeline_parametre.serve(name="pipeline-parametre")


Writing serveur_parametre.py


### À vous : D.2 lancer le déploiement depuis l'interface
1. Dans un terminal (le serveur reste lancé dans l'autre), tapez : `python serveur_parametre.py`. Laissez-le ouvert.
2. Dans l'interface, cliquez sur « Deployments ». Ouvrez `pipeline-parametre`.
3. Cliquez sur « Run », puis « Custom run ». Le champ `fichier` apparaît avec sa valeur par défaut.
4. Lancez une exécution avec `commandes_increment.csv`.
5. Lancez une seconde exécution avec `inexistant.csv`.

**Questions :**
1. Quel est l'état de chaque exécution ?
2. Où lisez-vous le paramètre utilisé par une exécution ?
3. Pourquoi la seconde exécution a-t-elle cet état ?

*Votre réponse :* 
1. La première est `Completed`, la seconde `Failed`.
2. Dans l'onglet « Parameters » de l'exécution.
3. Le fichier `inexistant.csv` n'existe pas : la tâche `controler_source` lève une erreur, et le pipeline ne démarre pas.

## Partie E : Lire et piloter une planification
### À vous : E.1 le déploiement planifié du TP 2
1. Arrêtez `python serveur_parametre.py` avec `Ctrl+C`.
2. Dans un terminal, tapez : `python serveur_prefect.py`. Ce script publie un flux lancé toutes les 30 secondes.
3. Dans l'interface, ouvrez « Deployments », puis `pipeline-toutes-les-30-secondes`.
4. Repérez la planification du déploiement.
5. Dans « Runs », filtrez l'état « Scheduled » : ce sont les exécutions prévues. Prefect en prépare une centaine à l'avance.
6. Mettez la planification en pause (interrupteur de la planification du déploiement). Attendez une minute : plus aucune exécution ne doit apparaître.
7. Remettez-la en marche.

**Questions :**
1. Quelle est la planification affichée ?
2. Que deviennent les exécutions « Scheduled » quand vous mettez la planification en pause ?
3. En pause, le déploiement existe-t-il encore ? Peut-on le lancer à la main ?

*Votre réponse :* 
1. Toutes les 30 secondes.
2. Elles disparaissent toutes : plus aucune exécution n'est prévue. À la reprise, une centaine de nouvelles exécutions « Scheduled » apparaissent.
3. Oui. Le déploiement existe toujours, et on peut le lancer à la main avec « Run ». Seul le lancement automatique est arrêté.

## Partie F : Garder l'historique des changements d'un fichier
Le script `surveillance_prefect.py` du TP 2 ne garde que la **dernière** empreinte du fichier, dans `derniere_empreinte.json`. On ne sait pas ce qui s'est passé avant.

Ici, on garde **chaque nouvelle empreinte**, avec la date et une mention. Le fichier s'appelle `historique_empreintes.json`. C'est une liste d'entrées, une par changement :

```
[
  {
    "date": "2026-11-05 12:33:34",
    "fichier": "data/commandes_increment.csv",
    "empreinte": "9e107d9d372bb6826bd81d3542a419d6",
    "mention": "premier passage"
  },
  {
    "date": "2026-11-05 12:41:10",
    "fichier": "data/commandes_increment.csv",
    "empreinte": "e4d909c290d0fb1ca068ffaddf22cbd0",
    "mention": "fichier modifié"
  }
]
```

La mention vaut :
- `premier passage` pour la première entrée de l'historique ;
- `fichier modifié` pour toutes les suivantes.

On n'écrit une entrée que si l'empreinte a changé. Un passage où le fichier est identique n'ajoute rien.

### À vous : F.1 écrire la tâche `fichier_a_change`
**À faire :** complétez le script `surveillance_historique.py`. La tâche `lancer_pipeline` et le flux `surveiller` sont donnés.

La tâche `fichier_a_change()` :
1. lit l'historique (une liste vide si le fichier n'existe pas) ;
2. calcule l'empreinte actuelle du fichier ;
3. renvoie `False`, sans rien écrire, si elle est égale à celle de la dernière entrée ;
4. sinon ajoute une entrée avec la date, le fichier, l'empreinte et la mention, puis écrit l'historique et renvoie `True`.

Indices : `datetime.now().strftime("%Y-%m-%d %H:%M:%S")` donne la date. `json.dump(..., indent=2)` écrit un JSON lisible.

In [13]:
%%writefile surveillance_historique.py
# Objectif : garder l'historique de toutes les empreintes du fichier, avec une mention.
# Résultat attendu : le fichier historique_empreintes.json contient une entrée à chaque changement du fichier, et aucune quand le fichier est identique.
import json
import os
import sqlite3
from datetime import datetime

from prefect import flow, get_run_logger, task
from prefect.runtime import flow_run

import etl
import outils

FICHIER = "data/commandes_increment.csv"
HISTORIQUE = "historique_empreintes.json"     # liste d'entrées : date, fichier, empreinte, mention


@task
def fichier_a_change():
    # Étape 1 : lire l'historique (une liste vide si le fichier HISTORIQUE n'existe pas)
    # Étape 2 : calculer l'empreinte actuelle du fichier
    # Étape 3 : si elle est égale à celle de la dernière entrée, renvoyer False sans rien écrire
    # Étape 4 : sinon, ajouter une entrée (date, fichier, empreinte, mention)
    # Étape 5 : écrire l'historique, puis renvoyer True
    historique = []
    if os.path.exists(HISTORIQUE):
        historique = json.load(open(HISTORIQUE, encoding="utf8"))
    actuelle = outils.empreinte(FICHIER)
    if historique and historique[-1]["empreinte"] == actuelle:
        return False
    mention = "fichier modifié" if historique else "premier passage"
    historique.append({"date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "fichier": FICHIER,
                       "empreinte": actuelle, "mention": mention})
    json.dump(historique, open(HISTORIQUE, "w", encoding="utf8"), ensure_ascii=False, indent=2)
    return True


@task
def lancer_pipeline():
    con = sqlite3.connect(etl.DB)
    etl.pipeline(con, os.path.basename(FICHIER), flow_run.name, declencheur="prefect")
    con.close()
    get_run_logger().info("Le pipeline a été lancé sur %s.", FICHIER)


@flow
def surveiller():
    if fichier_a_change():
        lancer_pipeline()
    else:
        get_run_logger().info("Le fichier n'a pas changé : le pipeline n'est pas lancé.")


if __name__ == "__main__":
    surveiller.serve(name="surveillance-historique", interval=10)


Writing surveillance_historique.py


### Démo : F.2 tester l'historique
La cellule suivante lance le flux trois fois, puis affiche l'historique. Entre le deuxième et le troisième passage, une commande est ajoutée au fichier.

In [14]:
# Cette cellule teste le script de l'historique. Il n'y a rien à compléter.
import importlib

import surveillance_historique
importlib.reload(surveillance_historique)      # recharge le script que vous venez d'écrire
from surveillance_historique import surveiller

if os.path.exists("historique_empreintes.json"):
    os.remove("historique_empreintes.json")    # on repart sans historique

print("--- Premier passage ---")
surveiller()
print("--- Deuxième passage : le fichier n'a pas changé ---")
surveiller()
nouvelle = outils.ajouter_commande_valide()
print("--- Troisième passage : la commande", nouvelle, "vient d'être ajoutée ---")
surveiller()

print()
print("Contenu de historique_empreintes.json :")
print(pd.DataFrame(json.load(open("historique_empreintes.json", encoding="utf8")))[["date", "empreinte", "mention"]])

--- Premier passage ---


13:30:09.201 | INFO    | Flow run 'kind-basilisk' - Beginning flow run 'kind-basilisk' for flow 'surveiller'

13:30:09.203 | INFO    | Flow run 'kind-basilisk' - View at http://127.0.0.1:4200/runs/flow-run/2ec84ab7-d08a-45eb-a62e-d6a428f99ba1

13:30:09.212 | INFO    | Task run 'fichier_a_change-5b6' - Finished in state Completed()

13:30:09.283 | INFO    | Task run 'lancer_pipeline-d36' - Le pipeline a été lancé sur data/commandes_increment.csv.

13:30:09.289 | INFO    | Task run 'lancer_pipeline-d36' - Finished in state Completed()

... (11 outputs left out)

13:30:11.878 | INFO    | Task run 'lancer_pipeline-769' - Le pipeline a été lancé sur data/commandes_increment.csv.

13:30:11.881 | INFO    | Task run 'lancer_pipeline-769' - Finished in state Completed()

13:30:12.831 | INFO    | Flow run 'impetuous-moose' - Finished in state Completed()


Contenu de historique_empreintes.json :
                  date                         empreinte          mention
0  2026-11-05 13:30:09  dc726c4b46668331b05c64955dea0058  premier passage
1  2026-11-05 13:30:11  dc1cd53af3ca10fe064aad29311e407b  fichier modifié


**Lecture :** l'historique contient **deux** entrées, alors qu'il y a eu trois passages.
- La première a la mention « premier passage ».
- La seconde a la mention « fichier modifié ».
- Le deuxième passage n'a rien ajouté : le fichier était identique.

Chaque exécution de cette démo ajoute une commande au fichier. Si vous la relancez, le contenu de l'historique change.

## Partie G : Publier un résumé dans l'interface
Le journal est long à lire. Prefect permet de publier un **artefact** : un petit tableau, affiché dans l'interface, sur la page de l'exécution.

On repère aussi les exécutions d'un flux avec une **étiquette** (*tag*) : un mot qu'on choisit, que l'interface permet de filtrer.

### À vous : G.1 publier le résumé et ajouter une étiquette
**À faire :** reprenez le script `surveillance_historique.py` (donné ci-dessous, avec votre solution de la partie F) et complétez deux endroits.
1. Dans `lancer_pipeline`, après le lancement du pipeline : publiez un tableau avec le fichier, le nombre de lignes lues et le nombre de lignes chargées.
   - Fonction : `create_markdown_artifact(key=..., markdown=..., description=...)`, importée depuis `prefect.artifacts`.
   - Le texte `markdown` est un tableau : une ligne d'en-têtes `| Fichier | Lues | Chargées |`, une ligne `|---|---|---|`, puis une ligne de valeurs.
   - La clé (`key`) est `resume-pipeline`.
2. Dans le bloc `__main__` : publiez le flux avec le nom `surveillance-historique`, un intervalle de 10 secondes et l'étiquette `surveillance`.
   - Option : `tags=["surveillance"]`.

In [31]:
%%writefile surveillance_historique.py
# Objectif : publier un résumé dans l'interface et repérer les exécutions avec une étiquette.
# Résultat attendu : chaque exécution qui lance le pipeline a un artefact « resume-pipeline » et l'étiquette « surveillance ».
import json
import os
import sqlite3
from datetime import datetime

from prefect import flow, get_run_logger, task
from prefect.artifacts import create_markdown_artifact
from prefect.runtime import flow_run

import etl
import outils

FICHIER = "data/commandes_increment.csv"
HISTORIQUE = "historique_empreintes.json"


@task
def fichier_a_change():
    historique = []
    if os.path.exists(HISTORIQUE):
        historique = json.load(open(HISTORIQUE, encoding="utf8"))
    actuelle = outils.empreinte(FICHIER)
    if historique and historique[-1]["empreinte"] == actuelle:
        return False
    mention = "fichier modifié" if historique else "premier passage"
    historique.append({"date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "fichier": FICHIER,
                       "empreinte": actuelle, "mention": mention})
    json.dump(historique, open(HISTORIQUE, "w", encoding="utf8"), ensure_ascii=False, indent=2)
    return True


@task
def lancer_pipeline():
    con = sqlite3.connect(etl.DB)
    lus, chargees, rejets, nb_dup = etl.pipeline(con, os.path.basename(FICHIER), flow_run.name, declencheur="prefect")
    con.close()
    get_run_logger().info("Le pipeline a été lancé sur %s.", FICHIER)
    # Étape 1 : publier un tableau (fichier, lignes lues, lignes chargées) avec la clé « resume-pipeline »
    create_markdown_artifact(
        key="resume-pipeline",
        markdown=f"| Fichier | Lues | Chargées |\n|---|---|---|\n| {FICHIER} | {lus} | {chargees} |",
        description="Résumé du dernier lancement du pipeline",
    )


@flow
def surveiller():
    if fichier_a_change():
        lancer_pipeline()
    else:
        get_run_logger().info("Le fichier n'a pas changé : le pipeline n'est pas lancé.")


if __name__ == "__main__":
    # Étape 2 : publier le flux (nom, intervalle de 10 secondes, étiquette « surveillance »)
    surveiller.serve(name="surveillance-historique", interval=10, tags=["surveillance"])


Overwriting surveillance_historique.py


### À vous : G.2 le voir dans l'interface
1. Arrêtez les scripts des parties D et E (`Ctrl+C`).
2. Dans un terminal, tapez : `python surveillance_historique.py`. Laissez-le ouvert.
3. Dans l'interface, ouvrez « Runs ». Une exécution de `surveiller` apparaît toutes les 10 secondes.
4. Ouvrez l'exécution qui a lancé le pipeline (celle qui a 2 tâches). Cliquez sur l'onglet « Artifacts ».
5. Ajoutez une commande au fichier avec la cellule ci-dessous, puis regardez la page « Runs » pendant 10 secondes.
6. Dans « Runs », filtrez avec l'étiquette `surveillance` (case « Tags »).

**Questions :**
1. Que contient l'artefact ?
2. Combien d'exécutions ont lancé le pipeline depuis le début, et pourquoi ?
3. À quoi sert l'étiquette ?

*Votre réponse :* 
1. Un tableau avec le fichier, les lignes lues et les lignes chargées.
2. Une seule au démarrage (premier passage), puis une par ajout de commande : l'historique n'écrit une entrée, et ne lance le pipeline, que si le fichier change.
3. À retrouver les exécutions de la surveillance parmi toutes les autres.

In [30]:
# Cette cellule ajoute une commande valide à commandes_increment.csv. Il n'y a rien à compléter.
nouvelle = outils.ajouter_commande_valide()
print("La commande", nouvelle, "vient d'être ajoutée à commandes_increment.csv.")

La commande 2063 vient d'être ajoutée à commandes_increment.csv.


## Bilan
Pour chaque situation, dites où vous regardez dans l'interface, et ce que vous y cherchez.
1. « Le pipeline n'a pas tourné cette nuit. »
2. « Le pipeline a échoué. Pourquoi ? »
3. « Je veux relancer le pipeline sur un autre fichier, tout de suite. »
4. « Je veux arrêter la planification quelques heures, sans rien supprimer. »
5. « Quand le fichier a-t-il changé pour la dernière fois ? »

*Votre réponse :* 
1. « Runs » : chercher les exécutions de la nuit. Il n'y en a pas : la planification est en pause, ou le script ne tournait plus.
2. « Runs », puis l'exécution à l'état `Failed` : onglet « Logs », et la tâche en échec.
3. « Deployments », puis « Run » et « Custom run » avec le fichier en paramètre.
4. « Deployments », puis l'interrupteur de la planification.
5. Pas dans l'interface : dans `historique_empreintes.json`, dernière entrée.